In [1]:
import qgauss as qg
import numpy as np

In [3]:
"""
We start by modelling the familiar case of dispersive readout of a qubit coupled to a single linear mode.
"""
# Start by defining the annihilation and creation operators for the linear mode, and the spin-z operator for the qubit, plus some identity operators.
a = qg.destroy()
ad = qg.create()
id = qg.identity_cvs()
sz = qg.sigmaz()
s0 = qg.identity_fls(2)

# Now some parameters.
chi_a = 2.75; lw_a = 6; nth_a = 0.33; eps_a = 6*np.exp(-0.5j*np.pi)

# Note, that when adding or subtracting scalars from an operator, that this is automatically treated as scalar*identity.
# Multiplication and division of operators by scalars is also supported.
# Combining objects acting on different Hilbert spaces is done using qg.tesnor(A1,A2,...,AN) or A1&A2&...&AN for short.
# Beware of the order of operations when using &, bracketing is usually necessary to avoid this.
# But in either case, the following Hamiltonians will be the same, Hdisp == Htest
H1disp = (chi_a*sz&(a*ad+1/2)) + (s0&(eps_a*ad + np.conj(eps_a)*a))
H1test = chi_a*qg.tensor(a*ad+1/2,sz) + qg.tensor(eps_a*ad + np.conj(eps_a)*a,s0)

# We can now construct the Lindbladian, passing the driven-dispersive Hamiltonian and 
# "collapse" or "jump" operators for a thermal bath coupled to the linear mode.
L1 = qg.lindbladian(H=H1disp, c_ops=[np.sqrt(lw_a*(nth_a+1))*a&s0, np.sqrt(lw_a*nth_a)*ad&s0])

# Th following removes small elements below the specified tolerance. Default is 1e-12.
L1.tidyup(1e-15)

In [4]:
# Now, we can calculate the backaction of the cavity on the |e><g| component of the qubit state.
# The first component is the total dephasing and frequency shift, both innate to the qubit and induced by the linear mode.
# The second component is the innate backaction, the third component is the measurement-induced backaction,
# and the fourth-component is the parasitic backaction.
qg.backaction_rate_steadystate(L1, qubit="e,g")

(np.complex128(8.008084745077454+9.483221992054577j),
 np.complex128(5.499999999999999j),
 np.complex128(7.209058819231679+0.378350567822038j),
 np.complex128(0.7990259258457748+3.604871424232539j))

In [47]:
# We can also calculate the backaction on the |g><e| component, and notice that only the requency shift changes sign from the above.
qg.backaction_rate_steadystate(L1, qubit="g,e")

(np.complex128(8.008084745077454-9.483221992054577j),
 np.complex128(-5.499999999999999j),
 np.complex128(7.209058819231679-0.378350567822038j),
 np.complex128(0.7990259258457748-3.604871424232539j))

In [48]:
# The exact measurement-induced dephasing is identitical within numerical error...
(2*(2*nth_a+1)*lw_a*(np.abs(eps_a)*chi_a)**2)/(((lw_a/2)**2-chi_a**2)**2 + (chi_a*lw_a*(2*nth_a+1))**2)

np.float64(7.209058819231684)

In [44]:
# ...as is the exact parasitic dephasing. For exact expressions see, Appendix B in Miller & Orr et al, arXiv:2603.12312 (2026)
# and Clerk & Utami, Phys. Rev. A 75 (2007).
(lw_a/2)*np.real(np.sqrt((1 + 2j*chi_a/lw_a)**2 + 8j*nth_a*chi_a/lw_a) - 1)

np.float64(0.7990259258457741)

In [49]:
# The diagonal elements naturally have no backaction
qg.backaction_rate_steadystate(L1, qubit="e,e")

(np.complex128(0j), np.complex128(0j), np.complex128(0j), np.complex128(0j))

In [50]:
# And if the qubit is not specified, the backaction on every component will be calculated.
qg.backaction_rate_steadystate(L1)

(array([[0.        +0.j        , 8.00808475-9.48322199j],
        [8.00808475+9.48322199j, 0.        +0.j        ]]),
 array([[0.+0.j , 0.-5.5j],
        [0.+5.5j, 0.+0.j ]]),
 array([[0.        +0.j        , 7.20905882-0.37835057j],
        [7.20905882+0.37835057j, 0.        +0.j        ]]),
 array([[0.        +0.j        , 0.79902593-3.60487142j],
        [0.79902593+3.60487142j, 0.        +0.j        ]]))

In [ ]:
"""
We now move on to the case where the driven mode is coupled via a beamsplitter to the mode which is strongly
coupled to the qubit. A smaller dispersive coupling between the driven mode and the qubit is also included.
Such a system will also give rise to a very weak dispersive energy exchange coupling betweeen the modes, which
is also included.
"""
# Here, we instead start off by defining the annhiliation operators and spin operators in the tensor basis, so we can
# just multiply the operators later when making the Hamiltonian rather than taking their tensor product. The Hermitian
# conjugate can also just be created by using the A.dag() method on the operator "A".
b = qg.tensor(qg.destroy(),qg.identity_cvs(),qg.identity_fls(2))
c = qg.tensor(qg.identity_cvs(),qg.destroy(),qg.identity_fls(2))
sz = qg.tensor(qg.identity_cvs(2),qg.sigmaz())
sm = qg.tensor(qg.identity_cvs(2),qg.sigmam())

# Note: as the number of linear modes grows, it can be useful to use the following function 
# qg.destroy(M,N) = qg.tensor(qg.identity_cvs(M-1),qg.destroy(),qg.identity_cvs(N-m))
# This creates a destruction operator for mode-M in an N-mode system, and is faster than using qg.tensor. The same
# arguments may also be passed to qg.create(), qg.position(), qg.momentum(), and qg.num().

# Parameters
chi_c = 2.754; chi_b = 0.236; chi_bc = 0.05
g_bc = 10; eps_b = 20; 
lw_b = 39; lw_c = 0.110
gamma_2 = 0.0245; gamma_1 = 0.038
n_b = 0.0030; n_c = 0.0081

# Now we can make the two-mode Lindbladian with two modes coupled via beam-splitter in the rotating frame.
H2bc = (chi_b*b.dag()*b*sz 
        + chi_c*c.dag()*c*sz 
        + chi_bc*sz*(b.dag()*c + b*c.dag())
        + g_bc*(b.dag()*c + c.dag()*b)
        + eps_b*(b + b.dag())
       )
# In this case, we instead create the Lindbladian by succesively adding coherent and dissipation superoperators.
# We could also use qg.dissipator(np.sqrt(lw_b*n_b)*b.dag()) instead of lw_b*n_b*qg.dissipator(b.dag()).
L2bc = (qg.coherent(H2bc) 
        + lw_b*(n_b+1)*qg.dissipator(b) 
        + lw_b*n_b*qg.dissipator(b.dag()) 
        + lw_c*(n_c+1)*qg.dissipator(c) 
        + lw_c*n_c*qg.dissipator(c.dag())
        + (gamma_2/2)*qg.dissipator(sz)
        + gamma_1*qg.dissipator(sm)
       )
L2bc.tidyup()

In [52]:
# The backaction in this case can be solved as usual.
qg.backaction_rate_steadystate(L2bc, qubit="e,g")

(np.complex128(14.055790108985551+9.30528792911189j),
 np.complex128(0.043499999999998096-2.9899999999999993j),
 np.complex128(14.003296946286529+9.29209253523346j),
 np.complex128(0.008993162699025073+3.0031953938784293j))

In [ ]:
# But, notice that by including the T1 process in L2bc, that we can now longer use the Gaussian moment
# method for the "g,g" diagonal element since it now couples to the excited state due to qubit decay.
qg.backaction_rate_steadystate(L2bc, qubit="g,g")

In [5]:
"""
We can now consider the case of multiplexed qubit readout, where two qubits are dispersively coupled to a single linear mode.
This allows for the modelling of all possible ZZ-interactions. We will also add amplification at the linear mode. This example 
is not phyiscal, but is simply provided to demonstrate the scope of systems this module can handle.
"""
d = qg.tensor(qg.destroy(),qg.identity_fls([2,2]))
sz1 = qg.tensor(qg.identity_cvs(),qg.sigmaz(),qg.identity_fls(2))
sz2 = qg.tensor(qg.identity_cvs(),qg.identity_fls(2),qg.sigmaz())

# Parameters
chi_1 = 2.754; chi_2 = 4.679
eps_d = 10; sqz = 2*np.exp(0.5j*np.pi)
lw_d = 50
zeta = 1
gz_1 = 0.0245; gz_2 = 0.038; gz_3 = 0.01; gz_4 = 0.01

# Now we define the Hamiltonian and Lindbladian, including both coherent and dissipative ZZ-interactions.
H3zz = ((chi_1*sz1 + chi_2*sz2)*(d.dag()*d + d*d.dag())/2
        + zeta*sz1*sz2 
        + eps_d*(d.dag() + d)
        + sqz*d.dag()**2 + np.conj(sqz)*d**2
       )
L3zz = (qg.coherent(H3zz) 
        + lw_d*qg.dissipator(d)
        + (gz_1/2)*qg.dissipator(sz1)
        + (gz_2/2)*qg.dissipator(sz2)
        + (gz_3/2)*qg.dissipator(sz1*sz2)
        + (gz_4/2)*qg.dissipator(sz1+sz2)
       )

In [8]:
# With the added qubit, there are backacton rates associated with each element of the 4-by-4 density operator.
qg.backaction_rate_steadystate(L3zz)[0]

array([[0.        +0.j        , 0.40826189-3.82977624j,
        0.16564304-1.44938435j, 0.94875108-9.10477353j],
       [0.40826189+3.82977624j, 0.        +0.j        ,
        0.12466226+2.42788067j, 0.16564304-5.44938435j],
       [0.16564304+1.44938435j, 0.12466226-2.42788067j,
        0.        +0.j        , 0.40826189-7.82977624j],
       [0.94875108+9.10477353j, 0.16564304+5.44938435j,
        0.40826189+7.82977624j, 0.        +0.j        ]])

In [62]:
# We can specify the exact term in the reduced qubit density matrix to get individual rates, such as the |ee><gg| term...
qg.backaction_rate_steadystate(L3zz, qubit="ee,gg")

(np.complex128(0.948751081387303+9.10477352922274j),
 np.complex128(0.10249999999999915+0j),
 np.complex128(0.7312703053447174+1.5460675799912222j),
 np.complex128(0.11498077604258646+7.558705949231518j))

In [63]:
# ...or the |eg><ge| term, for example.
qg.backaction_rate_steadystate(L3zz, qubit="eg,ge")

(np.complex128(0.1246622642305652-2.4278806692640478j),
 np.complex128(0.0625+0j),
 np.complex128(0.05216795316533994-0.4538174934088577j),
 np.complex128(0.009994311065225255-1.97406317585519j))